# 00.4 — IDE deep dive (foolproof setup for VS Code, JupyterLab, Classic)

Notebook [00.1](00.1_welcome.ipynb) gave a quick overview of the three editor options. This notebook is the **deep dive** — what to actually click, how to recognize when things go wrong, and what to do when the UI looks different from what the docs describe (it often does — VS Code and JupyterLab both update frequently).

After this notebook you should be able to:

* Pick an IDE that fits your workflow.
* Set it up from a clean install, step-by-step.
* Diagnose the four most common kernel/interpreter problems.
* Switch between IDEs without re-doing your venv.

**Read this once before you start editing notebooks; refer back to it whenever something doesn't behave the way you expect.**

## Section 1 — The mental model that ties them all together

All three IDEs do the same fundamental thing: they talk to a **Jupyter kernel** (a Python process) over a network protocol. The differences are just UI on top of that protocol.

Pictorially:

```
┌──────────────┐       ┌────────────────────┐       ┌────────────────────┐
│   You (UI)   │ <───> │  Jupyter kernel    │ <───> │  Your venv's       │
│ (any editor) │       │  (ipykernel proc)  │       │  Python + packages │
└──────────────┘       └────────────────────┘       └────────────────────┘
```

Three things must align:

1. **The editor knows where the venv lives.** ("interpreter selection")
2. **The kernel is started from that venv's Python.** ("kernel selection")
3. **The notebook file shows up in the editor.** (file browser)

When something goes wrong, it's almost always (1) or (2). Sections 2-4 walk through each IDE; Section 5 has the universal troubleshooting flowchart.

## Section 2 — VS Code

The smoothest experience for most users. One editor for both `.ipynb` and `.py`, with cross-references between them.

### 2.1 — Install the required extensions

You need **two** Microsoft extensions. Open the Extensions sidebar (the squares icon, or `Cmd/Ctrl+Shift+X`) and install:

* **Python** (`ms-python.python`) — provides interpreter detection.
* **Jupyter** (`ms-toolsai.jupyter`) — provides notebook UI + kernel management.

Both are free, both are first-party. Installing Python alone is not enough; the Jupyter extension is what renders `.ipynb` files.

**Verification:** after installing, both extensions should show as enabled in the Extensions sidebar; the functional check is opening any `.ipynb` file — it should render as a notebook with cells and a kernel picker, not as raw JSON. If it renders as JSON, restart VS Code.

### 2.2 — Open the project folder

`File → Open Folder...` and pick the `neural_data_decoding/` directory (not a subdirectory). This is important — VS Code detects the `.venv/` folder relative to the workspace root.

If you opened just the `notebooks/` subfolder, close it and re-open the parent. The venv won't be found otherwise.

### 2.3 — Select the Python interpreter (workspace-wide)

Before opening any notebook, set the workspace's default interpreter:

1. Open the command palette: `Cmd+Shift+P` (macOS) or `Ctrl+Shift+P` (Linux/Windows).
2. Type `Python: Select Interpreter` and press Enter.
3. Pick the entry that says `.venv/bin/python` (or `.venv\Scripts\python.exe` on Windows). VS Code labels it something like:

   ```
   Python 3.11.x ('.venv': venv)  .venv/bin/python
   ```

4. If `.venv` doesn't appear in the list, click "Enter interpreter path..." → "Find..." and navigate to `<repo>/.venv/bin/python` (or `<repo>/.venv/Scripts/python.exe` on Windows).

The bottom status bar should now show `Python 3.11 ('.venv': venv)`.

### 2.4 — Open a notebook and select the kernel

Click the notebook file in the Explorer sidebar. The editor shows it with the Jupyter UI.

In the top-right corner of the notebook view, there's a **Select Kernel** button (it sometimes reads `Select Kernel` and sometimes shows the current kernel's name).

1. Click it.
2. Pick **Python Environments...** from the dropdown.
3. Pick the same `.venv/bin/python` you chose in 2.3.

**If VS Code asks "do you want to install ipykernel?"** — say yes. It auto-runs `pip install ipykernel` into the venv so the kernel can start. (You already have it if you ran `pip install -e ".[dev]"` per [00.2](00.2_set_up_your_environment.ipynb) — ipykernel is in the dev extras, not the base dependencies — but VS Code prompts anyway if it can't detect it.)

### 2.5 — Run cells

Click any code cell. Three ways to run it:

* `Shift+Enter` — run the cell, move to the next.
* `Ctrl+Enter` / `Cmd+Enter` — run the cell, stay on it.
* The ▶ button on the left side of the cell.

The kernel "starts" on the first cell run. Watch for a small spinner in the status bar; the first run can take a few seconds while the kernel initializes.

### 2.6 — Common VS Code gotchas

**"Select Kernel" doesn't list `.venv`.** You skipped step 2.3 (interpreter selection) or VS Code didn't pick up the workspace folder. Re-run `Python: Select Interpreter` and pick `.venv/bin/python` first, then try `Select Kernel` again.

**Kernel selected but `import neural_data_decoding` fails.** The kernel is running the wrong Python OR `pip install -e .` was never run. Run the diagnostic below.

**Variable explorer is empty.** Look for a Variables button/icon in the notebook toolbar (its exact appearance moves between VS Code versions — the command palette's "Jupyter: Focus on Variables View" always works).

**Run All button looks different.** UI shifts between versions. Open the command palette and search "Jupyter: Run All" if you can't find the button.

In [ ]:
# Diagnostic — verify your kernel is the venv's Python.
# Run this in the first cell of any notebook to confirm.
import sys
print("sys.executable:", sys.executable)
# Should end in '.venv/bin/python' (macOS/Linux) or '.venv\\Scripts\\python.exe' (Windows).
# If it doesn't, re-do steps 2.3 and 2.4.

## Section 3 — JupyterLab

The classic experience. Best when you want the lab UI's file browser, multi-tab notebooks, or a Variable Inspector extension.

### 3.1 — Install JupyterLab into the venv

```bash
source .venv/bin/activate
pip install jupyterlab
```

This puts `jupyter-lab` (and `jupyter`) executables in `.venv/bin/`. **The venv MUST be activated** when you run `pip install jupyterlab` — otherwise JupyterLab lands in some other Python and can't see your project's packages.

### 3.2 — Launch JupyterLab

From the activated venv:

```bash
jupyter lab notebooks/
```

This:

1. Starts a notebook server on a free local port (typically 8888).
2. Opens your default browser at `http://localhost:8888/lab`.
3. Shows the `notebooks/` directory in the left sidebar.

**The browser tab is the UI; the kernel runs in your terminal.** If you close the terminal, the kernel dies. Use `jupyter lab` in a separate terminal you can leave open, or in a `tmux` / `screen` session.

### 3.3 — Open a notebook and verify the kernel

Click `00_orientation/00.1_welcome.ipynb` in the left sidebar. It opens in the main pane.

The kernel is selected automatically — JupyterLab uses the Python that launched it (your venv). The kernel name appears in the top-right of the notebook ("Python 3 (ipykernel)").

If you ran `jupyter lab` from outside the venv accidentally, the kernel will be using the wrong Python. Stop the server (`Ctrl+C` in the terminal), `source .venv/bin/activate`, and re-launch.

### 3.4 — Multiple kernels visible

If JupyterLab shows more than one Python kernel option (this happens when you've installed `ipykernel` into multiple Pythons), you can pin the venv's kernel explicitly:

```bash
source .venv/bin/activate
python -m ipykernel install --user --name=ndd-venv --display-name="Python 3 (ndd-venv)"
```

Now `Python 3 (ndd-venv)` appears as an option in the JupyterLab kernel switcher. Pick that one.

### 3.5 — Common JupyterLab gotchas

**Browser asks for a token.** When JupyterLab launches, the terminal prints a URL with a `?token=...` query string. Open that URL exactly as printed. Or copy the token and paste it into the prompt.

**"Permission denied" on `notebooks/` folder.** You launched from outside the repo root and the relative path can't be resolved. Use `cd <repo>` first, then `jupyter lab notebooks/`.

**Notebook says "Kernel not found"**. The kernelspec the notebook is configured for doesn't exist locally. Click the kernel name in the top-right and pick the available `Python 3 (ipykernel)` kernel.

**Closing the browser doesn't stop the kernel.** It just disconnects. The kernel keeps running in your terminal until you `Ctrl+C` there.

**You want to keep variables across browser refreshes.** Don't close the browser tab — refresh it. The kernel stays alive.

## Section 4 — Jupyter Notebook (the simpler UI)

Same kernel mechanics, simpler single-document interface. Useful if JupyterLab feels busy.

Note on versions: since 2023, `pip install notebook` installs **Notebook 7**, which is rebuilt on JupyterLab components — it looks close to the classic pre-2018 UI but is a modern app. If you specifically want the original classic interface, that lives on as `pip install nbclassic`.

### 4.1 — Install + launch

```bash
source .venv/bin/activate
pip install notebook
jupyter notebook notebooks/
```

The launch flow is the same as JupyterLab — server in the terminal, UI in the browser. The URL is `http://localhost:8888/tree`.

### 4.2 — Key differences from JupyterLab

* The sidebar is collapsible but not as feature-rich.
* No multi-tab notebook view — each notebook opens in a new browser tab.
* No built-in Variable Inspector. Use the `%whos` line magic or print variables manually.
* Kernel switcher is in the `Kernel` menu.

Everything else (kernel selection, gotchas) is identical to JupyterLab — same backend.

### 4.3 — When to pick Classic over Lab

* You're teaching a workshop and want the simplest possible UI.
* You hit a JupyterLab bug or extension incompatibility.
* You prefer the older multi-tab-per-notebook style.

## Section 5 — Universal troubleshooting flowchart

When a cell errors with `ModuleNotFoundError: neural_data_decoding`, follow this in order:

### Step 1 — verify the kernel is the venv's Python

In any code cell:

```python
import sys
print(sys.executable)
```

* Ends in `.venv/bin/python` (or `.venv\Scripts\python.exe` on Windows)? **→ skip to Step 3.**
* Ends in something else? **→ Step 2.**

### Step 2 — fix the kernel selection

* **VS Code:** Command palette → `Python: Select Interpreter` → pick `.venv/bin/python`. Then `Notebook: Select Kernel` → `.venv/bin/python` again.
* **JupyterLab / Classic:** Stop the server (`Ctrl+C`), `source .venv/bin/activate`, re-launch.

Then re-run Step 1 to confirm.

### Step 3 — verify the package is installed in that Python

```python
import subprocess
r = subprocess.run([sys.executable, "-m", "pip", "show", "neural_data_decoding"],
                   capture_output=True, text=True)
print(r.stdout or "NOT INSTALLED in this environment")
```

* Shows "Name: neural_data_decoding"? **→ skip to Step 4.**
* Errors with "not found"? Run in your shell:

  ```bash
  source .venv/bin/activate
  cd <repo>
  pip install -e .
  ```

### Step 4 — verify the specific name you're importing exists

```python
import neural_data_decoding
print(dir(neural_data_decoding))
```

If the symbol you wanted isn't in the dump, you likely have a typo (the most common case) or the symbol was renamed. Grep the codebase for the actual name:

```bash
grep -rn "your_symbol_name" src/
```

### Step 5 — restart the kernel and run all cells from the top

If everything looks right but you're still seeing stale errors, the kernel may be holding old state from a previous run. **Restart Kernel and Run All Cells** is the nuclear option:

* **VS Code:** click the ⟲ icon in the notebook toolbar, then ⏵⏵ to run all.
* **JupyterLab:** `Kernel → Restart Kernel and Run All Cells`.
* **Classic:** `Kernel → Restart & Run All`.

## Section 6 — Decision matrix

Use this to pick an IDE if you don't already have a strong preference:

| You want… | Use | Why |
|---|---|---|
| One editor for `.ipynb` + `.py` | VS Code | inline source navigation, F12 / Cmd+Click jump-to-definition works across both |
| Standard browser-based notebooks | JupyterLab | the classic experience; familiar to most data scientists |
| Minimal UI, fast | Classic Jupyter | simpler than Lab; lower learning curve |
| MATLAB-like (variable explorer + plots in panes) | Spyder | closest in feel, though doesn't natively run notebooks |
| Remote work over SSH | JupyterLab + SSH tunnel | mature remote story (`jupyter lab --no-browser` + port forward) |
| Free + offline + reproducible | any of the three | all work without internet once installed |

There's no wrong answer. **You can switch IDEs at any time without changing anything in the repo** — the notebook files are the same; only the editor differs.

## Section 7 — Hands-on exercise

Open this notebook in a SECOND IDE you haven't used yet. (If you used VS Code for it, switch to JupyterLab; if you used JupyterLab, switch to VS Code.)

Run the cell below in both, side by side. The output should be identical — kernel, not editor, determines behavior.

In [ ]:
import sys, neural_data_decoding
print(f"  Python: {sys.executable}")
print(f"  Package: {neural_data_decoding.__file__}")
print()
print("  If both IDEs print the same paths, your venv setup is solid.")

## Section 8 — Common errors

### "VS Code's Select Kernel option is missing"

The Jupyter extension isn't installed or didn't activate. Check `View → Extensions` and confirm `ms-toolsai.jupyter` is enabled. Restart VS Code.

### "JupyterLab opens but I can't see my notebooks"

Wrong launch directory. Use `jupyter lab notebooks/` (or `jupyter lab` from inside `notebooks/`) so the file browser starts there.

### "Cell runs but the output is from yesterday's kernel"

You're seeing cached output. Re-execute the cell with `Shift+Enter` to force a refresh.

### "I changed the venv but the kernel still uses the old one"

VS Code caches the kernel choice per workspace. Re-run `Notebook: Select Kernel` and pick again. JupyterLab/Classic: restart the server.

### "Two `Python 3` kernels appear in the picker"

Multiple Python installations have `ipykernel`. Pin the venv's kernel with a unique name (Section 3.4) so you can tell them apart.

### "Code completion / hover-help doesn't work in VS Code"

The Pylance extension provides this, and it depends on the interpreter being set. Re-do `Python: Select Interpreter`.

## Section 9 — Further reading

* [VS Code Jupyter docs](https://code.visualstudio.com/docs/datascience/jupyter-notebooks) — VS Code's official walkthrough.
* [JupyterLab user guide](https://jupyterlab.readthedocs.io/en/stable/user/interface.html) — the JupyterLab UI tour.
* [Classic Jupyter Notebook docs](https://jupyter-notebook.readthedocs.io/) — the original.
* [`ipykernel` docs](https://ipykernel.readthedocs.io/) — the kernel-protocol layer all three IDEs share.

Next up: **[00.5 the command line for MATLAB users](00.5_the_command_line_for_matlab_users.ipynb)** — the tool everything else in this curriculum runs through.